# Question 4 — Model Deployment & External Validation
## FinGuard Analytics — Consumer Complaints Escalation Prediction

**Student:** Alexander Kaisergruber (70299)   
**Course:** Machine Learning MSc  

---

This notebook fulfils the deployment requirement of **Question 4** as outlined in the assignment brief.  
It operates entirely independently of the training notebook: the saved pipeline (`70299_Pipeline.pkl`) is loaded
from disk, and the custom feature engineering function is imported from the accompanying `feature_engineering.py`
script. No model re-training is performed here.

### Steps Executed
| Step | Description |
|------|-------------|
| 1 | Install and verify dependencies from `70299_requirements.txt` |
| 2 | Import the feature engineering function from `feature_engineering.py` |
| 3 | Load the serialised pipeline from `70299_Pipeline.pkl` |
| 4 | Load the external validation dataset (`complaints_modeltesting100.csv`) |
| 5 | Generate binary predictions (0 = Not Disputed, 1 = Disputed) |
| 6 | Validate output format and summarise results |

## Step 1 — Dependencies

Before loading the model in a new environment, install required packages:
```bash
pip install -r 70299_requirements.txt
```
The cell below prints core library versions to confirm the environment matches the training environment.

In [2]:
import numpy as np
import pandas as pd
import pickle
import os
import sklearn

print(f"numpy      : {np.__version__}")
print(f"pandas     : {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print()
print("Environment check complete.")

numpy      : 2.4.2
pandas     : 2.3.3
scikit-learn: 1.7.1

Environment check complete.


## Step 2 — Import Feature Engineering Function

The pipeline internally uses a custom `FunctionTransformer` that wraps `feature_engineering()`.  
This function must be importable from `feature_engineering.py` at load time; otherwise `pickle.load`
will raise an `AttributeError`. Both this notebook and `feature_engineering.py` must reside in the
same working directory, or `feature_engineering.py` must be on the Python path.

In [3]:
from feature_engineering import feature_engineering

print("feature_engineering function imported successfully.")
print(f"  Source file: {os.path.abspath('feature_engineering.py')}")

feature_engineering function imported successfully.
  Source file: c:\Users\Alex\Desktop\Machine Learning\Individual_Assignment1\Kaisergruber_ML-Individual_Assignment\feature_engineering.py


## Step 3 — Load the Saved Pipeline

The trained pipeline (`70299_Pipeline.pkl`) was saved in the training notebook using `pickle.dump`.  
It encapsulates the full preprocessing stack — feature engineering, imputation, scaling, one-hot
encoding — together with the best-performing Random Forest classifier, so no manual preprocessing
is required before calling `.predict()`.

In [4]:
PICKLE_PATH = '70299_Pipeline.pkl'

with open(PICKLE_PATH, 'rb') as f:
    loaded_pipeline = pickle.load(f)

print(f"Pipeline loaded from : {os.path.abspath(PICKLE_PATH)}")
print(f"File size            : {os.path.getsize(PICKLE_PATH) / 1024:.1f} KB")
print()
print("Pipeline steps:")
for name, step in loaded_pipeline.steps:
    print(f"  {name:25s} -> {type(step).__name__}")

Pipeline loaded from : c:\Users\Alex\Desktop\Machine Learning\Individual_Assignment1\Kaisergruber_ML-Individual_Assignment\70299_Pipeline.pkl
File size            : 1068.1 KB

Pipeline steps:
  feature_engineering       -> FunctionTransformer
  preprocessor              -> ColumnTransformer
  model                     -> XGBClassifier


## Step 3b — Load the Optimal Decision Threshold

The optimal F1 threshold was determined via out-of-fold cross-validation on the
training set (see `70299_Complaints_Notebook.ipynb`, section 2.3). It is saved in
`70299_Threshold.json` so predictions here are identical to those produced during
training evaluation — no retraining or re-tuning is required.


In [ ]:
import json as _json

THRESHOLD_PATH = '70299_Threshold.json'
with open(THRESHOLD_PATH, 'r') as f:
    OPTIMAL_THRESHOLD = _json.load(f)['threshold']

print(f'Optimal threshold loaded: {OPTIMAL_THRESHOLD:.3f}')
print(f'  (Default sklearn threshold is 0.500)')


## Step 4 — Load the External Validation Dataset

The external validation set (`complaints_modeltesting100.csv`) has the same column structure as the
training data. The target column (`Consumer disputed?`) is dropped before prediction to prevent
data leakage and to replicate the real-world inference scenario where the label is unknown.

In [5]:
TEST_PATH = 'Assignment Information/complaints_modeltesting100.csv'

df_test = pd.read_csv(TEST_PATH, low_memory=False)

print(f"Validation set loaded : {os.path.abspath(TEST_PATH)}")
print(f"Shape                 : {df_test.shape[0]} rows x {df_test.shape[1]} columns")
print()
print("Columns present:")
print(df_test.columns.tolist())

Validation set loaded : c:\Users\Alex\Desktop\Machine Learning\Individual_Assignment1\Kaisergruber_ML-Individual_Assignment\Assignment Information\complaints_modeltesting100.csv
Shape                 : 100 rows x 18 columns

Columns present:
['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']


## Step 5 — Generate Binary Predictions

The target column and the unique complaint identifier are separated from the feature matrix before
calling the pipeline.  

- **0** → Not Disputed (complaint resolved without consumer escalation)  
- **1** → Disputed (complaint escalated to a formal consumer dispute)  

The `feature_engineering.py` script contains a safety guard that drops `Consumer disputed?` even if
it is accidentally left in `X`; however, it is explicitly excluded here as best practice.

In [6]:
# Columns to exclude from the feature matrix
COLS_TO_EXCLUDE = ['Consumer disputed?', 'Complaint ID']

X_val = df_test.drop(
    columns=[c for c in COLS_TO_EXCLUDE if c in df_test.columns]
)

# Generate predictions using the optimal threshold
# predict_proba gives the probability of class 1 (Disputed).
# Applying OPTIMAL_THRESHOLD instead of the default 0.5 maximises F1.
val_probs       = loaded_pipeline.predict_proba(X_val)[:, 1]
val_predictions = (val_probs >= OPTIMAL_THRESHOLD).astype(int)

print(f'Predictions generated  : {len(val_predictions)}')
print(f'Threshold applied      : {OPTIMAL_THRESHOLD:.3f}')
print(f'Unique output values   : {list(set(val_predictions.tolist()))}  (must be 0 and/or 1)')
print(f'Predicted dispute rate : {val_predictions.mean():.1%}')
print()
print('First 20 predictions:')
if 'Complaint ID' in df_test.columns:
    preview = pd.DataFrame({
        'Complaint ID'    : df_test['Complaint ID'].values[:20],
        'Prediction (0/1)': val_predictions[:20]
    })
else:
    preview = pd.DataFrame({'Prediction (0/1)': val_predictions[:20]})
print(preview.to_string(index=False))


Predictions generated  : 100
Unique output values   : [0 1]  (must be 0 and/or 1)
Predicted dispute rate : 57.0%

First 20 predictions:
 Complaint ID  Prediction (0/1)
      2643643                 0
      2717094                 1
      2695343                 0
      2667580                 1
      2401267                 1
      2760752                 0
      2650919                 1
      2767889                 0
      2471214                 0
      2669605                 1
      2643035                 0
      2586378                 0
      2668855                 1
      2683933                 1
      2657154                 1
      2469494                 1
      2753556                 0
      2692049                 0
      2399949                 0
      2664201                 1


## Step 6 — Output Validation

The following cell asserts that all predictions conform to the required binary format and
produces a concise summary of the prediction distribution.

In [7]:
# Assert binary output format
assert set(np.unique(val_predictions)).issubset({0, 1}), \
    "ERROR: Predictions contain values other than 0 and 1!"

n_total     = len(val_predictions)
n_disputed  = int(val_predictions.sum())
n_not_disp  = n_total - n_disputed

print("Output format validation: PASSED")
print()
print("Prediction Summary")
print("-" * 40)
print(f"  Total complaints evaluated : {n_total}")
print(f"  Predicted Not Disputed (0) : {n_not_disp:5d}  ({n_not_disp/n_total:.1%})")
print(f"  Predicted Disputed     (1) : {n_disputed:5d}  ({n_disputed/n_total:.1%})")
print()
print("The pipeline is functional and ready for deployment.")

Output format validation: PASSED

Prediction Summary
----------------------------------------
  Total complaints evaluated : 100
  Predicted Not Disputed (0) :    43  (43.0%)
  Predicted Disputed     (1) :    57  (57.0%)

The pipeline is functional and ready for deployment.
